# Phase 2 — Multimodal Representation Ablation

**Purpose:** determine which multimodal representation should be carried into the Graph Transformer stage.

We previously observed that the learned 64-D representation did not beat the direct FinBERT baseline. This experiment tests whether that was caused by an overly aggressive information bottleneck.

### Candidates

| Representation | Design |
|---|---|
| DIRECT | 14 Price + 8 Fundamentals + 768 FinBERT = 790-D |
| MM_64 | Price 64 + Fundamentals 64 + Text 64 → Fusion 64 |
| MM_128 | Price 64 + Fundamentals 64 + Text 128 → Fusion 128 |
| MM_TEXT256_FUSION128 | Price 64 + Fundamentals 64 + Text 256 → Fusion 128 |
| MM_256 | Price 64 + Fundamentals 64 + Text 256 → Fusion 256 |

Every candidate uses the **same downstream 5-day LSTM**. Representation selection uses validation only; the test set is evaluated once after selection.


## 1. Setup

In [ ]:
import os, sys, glob, json, random, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score

REPO_ROOT = "/content/capstone"
REPO_URL = "https://github.com/AdityaMelkote3004/capstone.git"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)
else:
    subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=False)

sys.path.insert(0, REPO_ROOT)
!pip -q install transformers pyarrow scikit-learn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Load and Clean the Modeling Dataset

This explicitly fixes the `Volume_Change` / price infinity issue discovered in the previous run.

In [ ]:
parquets = sorted(glob.glob(os.path.join(REPO_ROOT, "**", "*.parquet"), recursive=True))

preferred = os.path.join(REPO_ROOT, "dataset", "stocknet_final_modeling_set.parquet")
candidates = [p for p in parquets if "phase2" not in p.lower()]

if os.path.exists(preferred):
    BASE_PARQUET = preferred
elif candidates:
    BASE_PARQUET = candidates[0]
else:
    raise FileNotFoundError("Could not find the base modeling parquet.")

df = pd.read_parquet(BASE_PARQUET)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

PRICE_FEATURES = [
    "Return", "RSI_14", "MACD", "MACD_Signal", "MACD_Hist",
    "Volatility_5", "Volatility_20", "Price_MA5_Ratio",
    "Price_MA10_Ratio", "Price_MA20_Ratio", "Volume_Change",
    "HL_Spread", "MA_5", "MA_10"
]

FUNDAMENTAL_FEATURES = [
    "Revenue", "NetIncome", "TotalAssets", "TotalLiabilities",
    "StockholdersEquity", "EPS", "Cash", "ROA"
]

required = PRICE_FEATURES + FUNDAMENTAL_FEATURES + [
    "Target", "Ticker", "Date", "Company_Texts"
]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

# Remove infinities BEFORE normalization.
for col in PRICE_FEATURES + FUNDAMENTAL_FEATURES:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

for col in PRICE_FEATURES:
    df[col] = df[col].fillna(0.0)

for col in FUNDAMENTAL_FEATURES:
    df[col] = df.groupby("Ticker")[col].ffill().fillna(0.0)

print("Dataset:", df.shape)
print("Tickers:", df["Ticker"].nunique())

for cols, name in [(PRICE_FEATURES, "Price"), (FUNDAMENTAL_FEATURES, "Fundamentals")]:
    a = df[cols].to_numpy()
    print(name, "NaN:", np.isnan(a).sum(), "Inf:", np.isinf(a).sum())


## 3. Load Existing FinBERT Embeddings

If `company_embeddings.npy` exists and matches the dataset, it is reused. Otherwise the notebook generates it with FinBERT.

In [ ]:
EMB_PATH = os.path.join(REPO_ROOT, "company_embeddings.npy")

if os.path.exists(EMB_PATH):
    company_emb = np.load(EMB_PATH)
    assert company_emb.shape == (len(df), 768), (
        f"Embedding shape {company_emb.shape} does not match {(len(df), 768)}"
    )
    print("Loaded:", company_emb.shape)
else:
    from transformers import AutoTokenizer, AutoModel

    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    finbert = AutoModel.from_pretrained("ProsusAI/finbert").to(DEVICE)
    finbert.eval()

    texts = df["Company_Texts"].fillna("").astype(str).tolist()
    company_emb = np.zeros((len(texts), 768), dtype=np.float32)

    BATCH_SIZE = 32

    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start+BATCH_SIZE]
        valid = [i for i, t in enumerate(batch) if t.strip()]

        if valid:
            enc = tokenizer(
                [batch[i] for i in valid],
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            with torch.no_grad():
                out = finbert(**enc).last_hidden_state[:, 0, :]

            out = out.cpu().numpy().astype(np.float32)

            for j, local_i in enumerate(valid):
                company_emb[start + local_i] = out[j]

        if start % (BATCH_SIZE * 20) == 0:
            print(f"{min(start+BATCH_SIZE, len(texts))}/{len(texts)}")

    np.save(EMB_PATH, company_emb)

print("Embedding shape:", company_emb.shape)
print("NaN:", np.isnan(company_emb).sum())
print("Inf:", np.isinf(company_emb).sum())


## 4. Chronological Split + Train-Only Normalization

In [ ]:
TRAIN_END = pd.Timestamp("2015-03-31")
VAL_START = pd.Timestamp("2015-04-01")
VAL_END = pd.Timestamp("2015-07-31")
TEST_START = pd.Timestamp("2015-08-01")

train_mask = df["Date"] <= TRAIN_END
val_mask = (df["Date"] >= VAL_START) & (df["Date"] <= VAL_END)
test_mask = df["Date"] >= TEST_START

price_mean = df.loc[train_mask, PRICE_FEATURES].mean()
price_std = df.loc[train_mask, PRICE_FEATURES].std().replace(0, 1).fillna(1)

fund_mean = df.loc[train_mask, FUNDAMENTAL_FEATURES].mean()
fund_std = df.loc[train_mask, FUNDAMENTAL_FEATURES].std().replace(0, 1).fillna(1)

price_arr = ((df[PRICE_FEATURES] - price_mean) / price_std).fillna(0).to_numpy(np.float32)
fund_arr = ((df[FUNDAMENTAL_FEATURES] - fund_mean) / fund_std).fillna(0).to_numpy(np.float32)

train_emb = company_emb[train_mask.to_numpy()]
emb_mean = train_emb.mean(axis=0)
emb_std = train_emb.std(axis=0)
emb_std[emb_std < 1e-6] = 1.0

company_arr = ((company_emb - emb_mean) / emb_std).astype(np.float32)

print("Rows:")
print("Train:", int(train_mask.sum()))
print("Val:  ", int(val_mask.sum()))
print("Test: ", int(test_mask.sum()))

for name, arr in [("Price", price_arr), ("Fundamentals", fund_arr), ("Company", company_arr)]:
    print(name, "NaN:", np.isnan(arr).sum(), "Inf:", np.isinf(arr).sum())


## 5. Build Identical 5-Day Samples

In [ ]:
WINDOW = 5
work = df.copy()
work["_row_id"] = np.arange(len(work))

def build_samples(mask):
    frame = work.loc[mask]
    samples = []

    for ticker, group in frame.groupby("Ticker"):
        group = group.sort_values("Date").reset_index(drop=True)
        row_ids = group["_row_id"].to_numpy()

        for i in range(WINDOW, len(group)):
            rows = row_ids[i-WINDOW:i]

            samples.append({
                "price": price_arr[rows],
                "fund": fund_arr[rows],
                "company": company_arr[rows],
                "target": int(group.iloc[i]["Target"]),
                "ticker": ticker,
                "date": group.iloc[i]["Date"]
            })

    return samples

class MultiDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "price": torch.tensor(s["price"], dtype=torch.float32),
            "fund": torch.tensor(s["fund"], dtype=torch.float32),
            "company": torch.tensor(s["company"], dtype=torch.float32),
            "target": torch.tensor(s["target"], dtype=torch.long)
        }

train_ds = MultiDataset(build_samples(train_mask))
val_ds = MultiDataset(build_samples(val_mask))
test_ds = MultiDataset(build_samples(test_mask))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print("Train samples:", len(train_ds))
print("Val samples:  ", len(val_ds))
print("Test samples: ", len(test_ds))


## 6. Representation Candidates

The direct control preserves all 790 input values.

For learned representations, the modality encoders are trained **end-to-end with the LSTM**, so every candidate gets the same learning objective.

The question is not "is 64-D mathematically better?" It is:

> Which representation gives the best downstream validation performance when everything else is held constant?


In [ ]:
REPRESENTATIONS = {
    "DIRECT": {
        "type": "direct",
        "dim": 790
    },
    "MM_64": {
        "type": "learned",
        "text_dim": 64,
        "fusion_dim": 64
    },
    "MM_128": {
        "type": "learned",
        "text_dim": 128,
        "fusion_dim": 128
    },
    "MM_TEXT256_FUSION128": {
        "type": "learned",
        "text_dim": 256,
        "fusion_dim": 128
    },
    "MM_256": {
        "type": "learned",
        "text_dim": 256,
        "fusion_dim": 256
    }
}

class ModalityEncoder(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=None):
        super().__init__()
        hidden = hidden or max(out_dim, in_dim // 2)

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden),
            nn.Dropout(0.1),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

class FusionEncoder(nn.Module):
    def __init__(self, text_dim, fusion_dim):
        super().__init__()

        self.price_encoder = ModalityEncoder(14, 64)
        self.fund_encoder = ModalityEncoder(8, 64)
        self.text_encoder = ModalityEncoder(
            768, text_dim, hidden=max(256, text_dim * 2)
        )

        total_dim = 64 + 64 + text_dim
        hidden = max(fusion_dim, total_dim // 2)

        self.fusion = nn.Sequential(
            nn.Linear(total_dim, hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden),
            nn.Dropout(0.1),
            nn.Linear(hidden, fusion_dim)
        )

    def forward(self, price, fund, company):
        zp = self.price_encoder(price)
        zf = self.fund_encoder(fund)
        zt = self.text_encoder(company)
        return self.fusion(torch.cat([zp, zf, zt], dim=-1))


## 7. Common LSTM + Evaluation

In [ ]:
class RepresentationLSTM(nn.Module):
    def __init__(self, config):
        super().__init__()

        if config["type"] == "direct":
            self.encoder = None
            input_dim = config["dim"]
        else:
            self.encoder = FusionEncoder(
                config["text_dim"],
                config["fusion_dim"]
            )
            input_dim = config["fusion_dim"]

        self.lstm = nn.LSTM(
            input_dim,
            64,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2)
        )

    def forward(self, price, fund, company):
        B, W, _ = price.shape

        if self.encoder is None:
            z = torch.cat([price, fund, company], dim=-1)
        else:
            z = self.encoder(
                price.reshape(B * W, -1),
                fund.reshape(B * W, -1),
                company.reshape(B * W, -1)
            ).reshape(B, W, -1)

        _, (h_n, _) = self.lstm(z)
        return self.classifier(h_n[-1])

def evaluate(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch["price"].to(DEVICE),
                batch["fund"].to(DEVICE),
                batch["company"].to(DEVICE)
            )

            y_true.extend(batch["target"].numpy())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())
            y_prob.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "auc": auc
    }


## 8. Train One Candidate

In [ ]:
def train_candidate(name, config, epochs=40, patience=7):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    set_seed()

    model = RepresentationLSTM(config).to(DEVICE)
    print("Parameters:", sum(p.numel() for p in model.parameters()))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()

    best_mcc = -1e9
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()

        for batch in train_loader:
            optimizer.zero_grad()

            logits = model(
                batch["price"].to(DEVICE),
                batch["fund"].to(DEVICE),
                batch["company"].to(DEVICE)
            )

            loss = criterion(logits, batch["target"].to(DEVICE))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        val = evaluate(model, val_loader)

        if val["mcc"] > best_mcc:
            best_mcc = val["mcc"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Ep {epoch:3d} | "
                f"Val Acc={val['accuracy']:.3f} "
                f"F1={val['f1']:.3f} "
                f"MCC={val['mcc']:.4f} "
                f"AUC={val['auc']:.4f}"
            )

        if wait >= patience:
            print("Early stop at epoch", epoch)
            break

    model.load_state_dict(best_state)
    return model, evaluate(model, val_loader)


## 9. Run the Representation Ablation

In [ ]:
models = {}
val_results = {}

for name, config in REPRESENTATIONS.items():
    models[name], val_results[name] = train_candidate(name, config)

print("\n" + "=" * 90)
print("VALIDATION RESULTS")
print("=" * 90)
print(f"{'Representation':<30} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 75)

for name, m in val_results.items():
    print(
        f"{name:<30} "
        f"{m['accuracy']:>8.4f} "
        f"{m['f1']:>8.4f} "
        f"{m['mcc']:>8.4f} "
        f"{m['auc']:>8.4f}"
    )


## 10. Select the Representation Using Validation Only

In [ ]:
ranking = sorted(
    val_results.items(),
    key=lambda item: (item[1]["mcc"], item[1]["auc"]),
    reverse=True
)

print("Validation ranking:")
for i, (name, m) in enumerate(ranking, 1):
    print(
        f"{i}. {name} | "
        f"MCC={m['mcc']:.4f} | "
        f"AUC={m['auc']:.4f}"
    )

BEST_NAME = ranking[0][0]
print("\nSelected representation:", BEST_NAME)


## 11. Final Test Evaluation

In [ ]:
best_test = evaluate(models[BEST_NAME], test_loader)

print("=" * 80)
print("FINAL TEST RESULT")
print("=" * 80)
print("Representation:", BEST_NAME)

for k, v in best_test.items():
    print(f"{k.upper():>10}: {v:.4f}")


## 12. Existing Phase 2A Reference Results

In [ ]:
baseline = pd.DataFrame([
    ["Price + Company FinBERT", "Logistic Regression", 0.4920, 0.5589, -0.0080, 0.4953],
    ["Price + Company FinBERT", "LSTM",                0.5248, 0.4721,  0.0459, 0.5304],
    ["Price + Company FinBERT", "MLP",                 0.4830, 0.6109, -0.0221, 0.5053],
    ["Price + Fundamentals + Company FinBERT", "Logistic Regression", 0.4932, 0.5606, -0.0053, 0.4961],
    ["Price + Fundamentals + Company FinBERT", "LSTM",                0.4870, 0.6550,  0.0000, 0.5270],
    ["Price + Fundamentals + Company FinBERT", "MLP",                 0.5038, 0.5506,  0.0140, 0.5124],
], columns=["Feature Set", "Model", "Accuracy", "F1", "MCC", "AUC"])

baseline


## 13. Save Results + Generate the Markdown Report

In [ ]:
RESULT_DIR = os.path.join(
    REPO_ROOT,
    "results",
    "phase2_representation_ablation"
)
os.makedirs(RESULT_DIR, exist_ok=True)

val_df = pd.DataFrame([
    [name, m["accuracy"], m["f1"], m["mcc"], m["auc"]]
    for name, m in val_results.items()
], columns=[
    "Representation", "Val_Accuracy", "Val_F1", "Val_MCC", "Val_AUC"
]).sort_values(
    ["Val_MCC", "Val_AUC"],
    ascending=False
)

val_df.to_csv(
    os.path.join(RESULT_DIR, "validation_ablation.csv"),
    index=False
)

results = {
    "phase": "Phase 2",
    "experiment": "Multimodal Representation Ablation",
    "window_size": WINDOW,
    "selection_rule": "Validation MCC, then validation AUC",
    "selected_representation": BEST_NAME,
    "validation_results": val_results,
    "final_test_result": best_test
}

with open(os.path.join(RESULT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

report = [
    "# Phase 2 — Multimodal Representation Ablation",
    "",
    "## Goal",
    "",
    "This experiment tests whether the previous 64-D multimodal bottleneck was too aggressive.",
    "",
    "## Candidates",
    "",
    "| Representation | Design |",
    "|---|---|",
    "| DIRECT | Price + Fundamentals + 768-D FinBERT |",
    "| MM_64 | Price 64 + Fundamentals 64 + Text 64 → Fusion 64 |",
    "| MM_128 | Price 64 + Fundamentals 64 + Text 128 → Fusion 128 |",
    "| MM_TEXT256_FUSION128 | Price 64 + Fundamentals 64 + Text 256 → Fusion 128 |",
    "| MM_256 | Price 64 + Fundamentals 64 + Text 256 → Fusion 256 |",
    "",
    "## Validation Results",
    "",
    "| Representation | Accuracy | F1 | MCC | AUC |",
    "|---|---:|---:|---:|---:|"
]

for _, row in val_df.iterrows():
    report.append(
        f"| {row['Representation']} | "
        f"{row['Val_Accuracy']:.4f} | "
        f"{row['Val_F1']:.4f} | "
        f"{row['Val_MCC']:.4f} | "
        f"{row['Val_AUC']:.4f} |"
    )

report += [
    "",
    "## Selected Representation",
    "",
    f"**{BEST_NAME}** was selected using validation MCC, with validation AUC as the secondary criterion.",
    "",
    "## Final Test Result",
    "",
    "| Metric | Value |",
    "|---|---:|"
]

for k, v in best_test.items():
    report.append(f"| {k.upper()} | {v:.4f} |")

report += [
    "",
    "## Existing Phase 2A Reference",
    "",
    "| Feature Set | Model | Accuracy | F1 | MCC | AUC |",
    "|---|---|---:|---:|---:|---:|"
]

for _, r in baseline.iterrows():
    report.append(
        f"| {r['Feature Set']} | {r['Model']} | "
        f"{r['Accuracy']:.4f} | {r['F1']:.4f} | "
        f"{r['MCC']:.4f} | {r['AUC']:.4f} |"
    )

report += [
    "",
    "## Interpretation",
    "",
    "Representation quality is judged empirically. A richer representation is preferred only if it improves validation performance under the same downstream LSTM setup. The test set is not used for architecture selection.",
    "",
    "## Next Step",
    "",
    "The selected representation becomes the candidate stock-node representation for the centralized Stock Graph + Graph Transformer experiment. Global-event conditioning and federated learning should be added only after the centralized graph baseline is established.",
    ""
]

report_path = os.path.join(
    RESULT_DIR,
    "phase2_representation_ablation.md"
)

Path(report_path).write_text("\n".join(report), encoding="utf-8")

print("Saved results to:", RESULT_DIR)
print("Markdown:", report_path)


## 14. Interpretation Guide

At the end:

- **MM_64 wins:** the compact representation is sufficient.
- **MM_128 / MM_256 wins:** the 64-D bottleneck discarded useful information.
- **DIRECT wins:** learned compression/fusion is not helping.
- **Several are close:** prefer the smaller representation for efficiency, then verify it in the graph experiment.

### Important

Do not repeatedly choose architectures using test performance.

**Validation chooses. Test evaluates.**

Once the representation is frozen, the next experiment is the centralized **Stock Graph + Graph Transformer**.
